# Notebook 02 — SFT Dataset B (Nemotron SFT Math)
**Assignment 04 | Track 2, Option D | NLP with Deep Learning — IBA**

**Model:** `Qwen/Qwen3-0.6B` — same model as notebook 01  
**Dataset:** `nvidia/Nemotron-SFT-Math-v3` — large math reasoning dataset, use ~2,160-row subset

Same model + same 5-trial ablation as notebook 01. **Only the dataset differs.**  
This means any performance difference between best_A and best_B = purely the dataset's effect.

**Run this on:** Any CUDA GPU machine (local 3090, Kaggle T4, Colab)  
**Prerequisite:** Run `00_baseline.ipynb` first to generate `prompts.json` and `gold_answers.json`

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Unsloth: auto-detects CUDA version and installs the correct build
!pip install -q "unsloth[cu124-torch260] @ git+https://github.com/unslothai/unsloth.git" 2>/dev/null || \
 pip install -q unsloth
!pip install -q trl peft transformers datasets accelerate bitsandbytes huggingface_hub python-dotenv openai

In [ ]:
import os, json, time, re, torch, pandas as pd
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from openai import OpenAI
from transformers import AutoTokenizer
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN    = os.environ["HF_TOKEN"]   # loaded from .env
BASE_MODEL  = "Qwen/Qwen3-0.6B"          # same as notebook 01 — cleaner dataset comparison
DATASET_ID  = "nvidia/Nemotron-SFT-Math-v3"
JUDGE_MODEL = "moonshotai/Kimi-K2-Instruct:novita"
JUDGE_CLIENT = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN,
)
MAX_SEQ_LEN = 2048
SUBSET_SIZE = 2160   # match Dataset A size for fair comparison
OUTPUT_DIR  = "./adapters_datasetB"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## Step 1 — Load Dataset, Inspect Columns, Handle Format

Nemotron column format is not confirmed — this cell detects it automatically.

In [ ]:
ds_full = load_dataset(DATASET_ID, split="train")
print(f"Full dataset size: {len(ds_full)} rows")
print(f"Columns: {list(ds_full.features.keys())}")
print("\nSample row (first 200 chars per field):")
for k, v in ds_full[0].items():
    print(f"  {k}: {str(v)[:200]}")

In [ ]:
# ── Auto-detect column format ─────────────────────────────────────────────────
sample = ds_full[0]
columns = list(sample.keys())

if "messages" in columns:
    DATASET_FORMAT = "messages"
    print("Detected: messages-list format")
elif "problem" in columns and "solution" in columns:
    DATASET_FORMAT = "flat"
    print("Detected: flat problem/solution format")
elif "input" in columns and "output" in columns:
    DATASET_FORMAT = "input_output"
    print("Detected: input/output format")
else:
    print(f"UNKNOWN format — columns: {columns}")
    print("You may need to manually define format_row below.")
    DATASET_FORMAT = "unknown"

# ── Check for TIR (Python tool-integrated reasoning) ─────────────────────────
# TIR solutions contain Python code blocks — filter them out for clean SFT
def has_python_code(text):
    return "```python" in str(text) or "import " in str(text)

sample_solution = str(list(sample.values())[1])  # check second field
print(f"\nSample solution contains Python code: {has_python_code(sample_solution)}")
print("(If True, will filter TIR rows — CoT-only kept)")

## Step 2 — Format Dataset

In [ ]:
tokenizer_for_format = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)

def format_row(row):
    """Format Nemotron row into chat-template string.
    Handles messages-list and flat problem/solution formats.
    """
    if DATASET_FORMAT == "messages":
        messages = row["messages"]
        # Prepend system message if not present
        if messages[0]["role"] != "system":
            messages = [{"role": "system",
                         "content": "You are a helpful math reasoning assistant. Think step by step."}] + messages
        return tokenizer_for_format.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )

    elif DATASET_FORMAT == "flat":
        problem  = row.get("problem", row.get("question", "")).strip()
        solution = row.get("solution", row.get("answer", "")).strip()
        messages = [
            {"role": "system",    "content": "You are a helpful math reasoning assistant. Think step by step."},
            {"role": "user",      "content": problem},
            {"role": "assistant", "content": solution},
        ]
        return tokenizer_for_format.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )

    elif DATASET_FORMAT == "input_output":
        messages = [
            {"role": "system",    "content": "You are a helpful math reasoning assistant. Think step by step."},
            {"role": "user",      "content": row["input"].strip()},
            {"role": "assistant", "content": row["output"].strip()},
        ]
        return tokenizer_for_format.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    else:
        raise ValueError(f"Unknown format: {DATASET_FORMAT}. Update format_row manually.")


def is_cot_row(row):
    """Return True if row is CoT-only (no Python TIR code)."""
    text = " ".join(str(v) for v in row.values())
    return not has_python_code(text)


# Filter to CoT-only rows, then take SUBSET_SIZE
ds_cot = ds_full.filter(is_cot_row)
print(f"After CoT filter: {len(ds_cot)} rows (removed {len(ds_full)-len(ds_cot)} TIR rows)")

ds_sub = ds_cot.shuffle(seed=42).select(range(min(SUBSET_SIZE, len(ds_cot))))
print(f"Using subset: {len(ds_sub)} rows")

ds_formatted = ds_sub.map(lambda row: {"text": format_row(row)})
print("\nSample formatted text (first 500 chars):")
print(ds_formatted[0]["text"][:500])

## Step 3 — Load Prompts and Judge Helper

In [ ]:
with open("prompts.json") as f:
    PROMPTS = json.load(f)
with open("gold_answers.json") as f:
    gold_answers = json.load(f)

JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator of reasoning quality.

QUESTION:
{question}

REFERENCE ANSWER (Gold):
{gold}

CANDIDATE RESPONSE:
{response}

Evaluate the candidate response on REASONING QUALITY:
5 = Correct answer, clear step-by-step reasoning, no errors
4 = Correct answer, reasoning mostly clear with minor gaps
3 = Partially correct or reasoning has notable gaps
2 = Wrong answer but shows some reasoning attempt
1 = Wrong answer, no meaningful reasoning

Respond in this exact format (nothing else):
SCORE: <number 1-5>
REASON: <one sentence justification>"""

def _parse_judge_output(out):
    score_match = re.search(r"SCORE\s*:\s*([1-5])", out, re.IGNORECASE)
    reason_match = re.search(r"REASON\s*:\s*(.+)", out, re.IGNORECASE | re.DOTALL)
    if not score_match:
        raise ValueError(f"Judge response did not include SCORE 1-5. Raw output: {out!r}")
    reason = reason_match.group(1).strip() if reason_match else out.strip()
    return int(score_match.group(1)), reason


def judge_response(question, gold, response):
    prompt = JUDGE_PROMPT_TEMPLATE.format(question=question, gold=gold, response=response)
    last_error = None
    for attempt in range(3):
        try:
            completion = JUDGE_CLIENT.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=120,
                temperature=0.0,
            )
            out = completion.choices[0].message.content.strip()
            return _parse_judge_output(out)
        except Exception as error:
            last_error = error
            time.sleep(2 * (attempt + 1))
    return 0, f"ERROR after retries: {last_error}"



def _apply_chat_template(tokenizer, messages):
    """Handles BatchEncoding (transformers 4.50+) and plain tensor (older)."""
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    )
    return encoded["input_ids"] if not isinstance(encoded, torch.Tensor) else encoded


def eval_model_on_prompts(model, tokenizer, trial_name):
    FastLanguageModel.for_inference(model)
    records = []
    for p in PROMPTS:
        messages = [
            {"role": "system",  "content": "You are a helpful math reasoning assistant. Think step by step."},
            {"role": "user",    "content": p["prompt"]}
        ]
        input_ids = _apply_chat_template(tokenizer, messages).to(model.device)
        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=512, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        score, reason = judge_response(p["prompt"], gold_answers[p["id"]], response)
        records.append({
            "trial": trial_name, "prompt_id": p["id"], "type": p["type"],
            "response": response, "score": score, "reason": reason
        })
        print(f"  [{trial_name}] {p['id']}: score={score}")
        time.sleep(0.5)
    return records

## Step 4 — Trial Loop (5 Trials — identical configs to notebook 01)

In [ ]:
TRIAL_CONFIGS = [
    {"name": "T1", "data_pct": 0.25, "rank": 16,
     "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
     "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
     "purpose": "Data efficiency — 25%"},
    {"name": "T2", "data_pct": 0.50, "rank": 16,
     "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
     "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
     "purpose": "Data efficiency — 50%"},
    {"name": "T3", "data_pct": 1.00, "rank": 16,
     "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
     "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
     "purpose": "Data efficiency — 100%"},
    {"name": "T4", "data_pct": 1.00, "rank": 32,
     "target_modules": ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
     "lr": 1e-4, "batch": 4, "grad_accum": 4, "epochs": 2,
     "purpose": "LoRA capacity — rank 32, wider modules"},
    {"name": "T5", "data_pct": 1.00, "rank": 64,
     "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
     "lr": 5e-5, "batch": 2, "grad_accum": 8, "epochs": 2,
     "purpose": "LoRA capacity — rank 64, deeper"},
]

# fp16/bf16 — only set when actually on GPU; CPU training uses full fp32
use_fp16 = torch.cuda.is_available() and not torch.cuda.is_bf16_supported()
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"fp16={use_fp16} | bf16={use_bf16}")

all_records   = []
trial_summary = []

for cfg in TRIAL_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Trial {cfg['name']}: {cfg['purpose']}")
    print('='*60)

    n = int(len(ds_formatted) * cfg["data_pct"])
    ds_train = ds_formatted.select(range(n))
    split    = ds_train.train_test_split(test_size=0.1, seed=42)
    train_ds, eval_ds = split["train"], split["test"]
    print(f"  Train: {len(train_ds)} | Eval: {len(eval_ds)}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True, token=HF_TOKEN
    )
    model = FastLanguageModel.get_peft_model(
        model, r=cfg["rank"],
        target_modules=cfg["target_modules"],
        lora_alpha=cfg["rank"],
        lora_dropout=0.05, bias="none",
        use_gradient_checkpointing=True,
    )

    adapter_path = os.path.join(OUTPUT_DIR, cfg["name"])

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=eval_ds,
        args=SFTConfig(
            output_dir=adapter_path,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            per_device_train_batch_size=cfg["batch"],
            gradient_accumulation_steps=cfg["grad_accum"],
            learning_rate=cfg["lr"],
            num_train_epochs=cfg["epochs"],
            eval_strategy="epoch",
            save_strategy="epoch",
            # load_best_model_at_end NOT used — incompatible with Unsloth 4-bit quantized models
            fp16=use_fp16,
            bf16=use_bf16,
            logging_steps=10,
            report_to="none",
        ),
    )

    train_result = trainer.train()
    val_loss     = trainer.evaluate()["eval_loss"]

    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"  Adapter saved to: {adapter_path}")
    print(f"  Train loss: {train_result.training_loss:.4f} | Val loss: {val_loss:.4f}")

    records = eval_model_on_prompts(model, tokenizer, cfg["name"])
    for r in records:
        r.update({"model": BASE_MODEL, "stage": "sft_datasetB",
                  "data_pct": cfg["data_pct"], "rank": cfg["rank"],
                  "lr": cfg["lr"], "val_loss": val_loss})
    all_records.extend(records)

    avg_score = sum(r["score"] for r in records) / len(records)
    trial_summary.append({
        "trial": cfg["name"], "purpose": cfg["purpose"],
        "data_pct": f"{cfg['data_pct']*100:.0f}%",
        "rank": cfg["rank"], "lr": cfg["lr"],
        "batch": cfg["batch"], "epochs": cfg["epochs"],
        "val_loss": round(val_loss, 4),
        "avg_judge_score": round(avg_score, 2),
        "adapter_path": adapter_path,
    })
    print(f"  Avg judge score: {avg_score:.2f}")

    del model, trainer
    torch.cuda.empty_cache()

print("\nAll 5 trials complete.")

## Step 5 — Select Best Trial and Save

In [ ]:
df_summary = pd.DataFrame(trial_summary)
print("\n── Trial Results ──")
print(df_summary[["trial","purpose","data_pct","rank","lr","epochs","val_loss","avg_judge_score"]].to_string(index=False))

best = df_summary.sort_values(
    ["avg_judge_score", "val_loss"], ascending=[False, True]
).iloc[0]

print(f"\n── Best Trial: {best['trial']} ──")
print(f"  Purpose:         {best['purpose']}")
print(f"  Avg judge score: {best['avg_judge_score']}")
print(f"  Val loss:        {best['val_loss']}")

df_summary.to_csv("trial_results_datasetB.csv", index=False)
pd.DataFrame(all_records).to_csv("all_scores_datasetB.csv", index=False)
with open("best_trial_datasetB.json", "w") as f:
    json.dump(best.to_dict(), f, indent=2)

print("\nSaved: trial_results_datasetB.csv, all_scores_datasetB.csv, best_trial_datasetB.json")